# RAG on Azure — Day 10: Parent-Document Retrieval (Small-to-Big) 

**Use case:** the right *short* chunk has been retrieved, but it doesn't contain enough context for a complete, nuanced answer. The exception is in the next paragraph. The qualification is two sentences away. Small-chunk RAG gives a technically-correct-but-practically-wrong answer.


- **Two-level chunking.** Each document is split into large **parents** (full sections, ~1500 chars) and small **children** (~400 chars). Embedding and retrieval happen on children; generation uses the parents.
- A new index schema with `parent_id` and `parent_content` fields alongside the usual content + vector.
- One new helper, `retrieve_parents()`, that retrieves child chunks, dedupes by parent, and returns the *parent sections* in a shape that drops into the existing `answer()` function.
- A demo where the same question yields a partial answer with small chunks but a complete, nuanced answer with parents.

## 0. Install dependencies

In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Setup

In [5]:
import sys
sys.path.insert(0, "..")

from common import load_config, get_openai_client, get_search_client, get_search_index_client
from common import chunk_text, load_pdfs, extract_sections
from common import embed_texts, retrieve, retrieve_parents, answer
from pathlib import Path

config = load_config(Path(r"C:\Users\sunil\OneDrive\Documents\GitHub\Basic RAG\RAG\.env"))
openai = get_openai_client(config)
index_client = get_search_index_client(config)
INDEX_NAME = "rag-policies-day10"
print("Setup OK. Index:", INDEX_NAME)

Setup OK. Index: rag-policies-day10


## 2. Load and two-level chunk

Each **section** is a parent. Each parent is split into multiple **children**. Every child record carries its parent's full text so we can retrieve children but generate from parents.

In [6]:
from pathlib import Path
import re

DATA_DIR = Path("../data-day10")
print("Looking in:", DATA_DIR.resolve())
raw_docs = load_pdfs(DATA_DIR)
assert raw_docs, f"No PDFs in {DATA_DIR.resolve()}"

CHILD_SIZE    = 400
CHILD_OVERLAP = 60

records = []
for d in raw_docs:
    sections = extract_sections(d["text"]) or [("Body", d["text"])]
    for section_title, section_body in sections:
        parent_id = f"{d['source']}::{section_title}"
        safe_pid  = re.sub(r"[^A-Za-z0-9_-]", "_", parent_id)
        for c_idx, child in enumerate(chunk_text(section_body, chunk_size=CHILD_SIZE, overlap=CHILD_OVERLAP)):
            records.append({
                "id":             f"{safe_pid}-c{c_idx}",
                "content":        child,           # small chunk -> embedded + retrieved
                "parent_id":      parent_id,
                "parent_content": section_body,    # large parent -> sent to LLM
                "source":         d["source"],
                "section":        section_title,
            })

parents = {(r["source"], r["section"]): r["parent_content"] for r in records}
print(f"\n{len(records)} children across {len(parents)} parent sections.")
print(f"Avg child size:  {sum(len(r['content']) for r in records)//len(records)} chars")
print(f"Avg parent size: {sum(len(p) for p in parents.values())//len(parents)} chars")
print("\nExample (one child + its parent):")
print(f"  CHILD  ({len(records[0]['content'])} chars):  {records[0]['content'][:120]}...")
print(f"  PARENT ({len(records[0]['parent_content'])} chars): {records[0]['parent_content'][:120]}...")

Looking in: C:\Users\sunil\OneDrive\Documents\GitHub\Basic RAG\RAG\data-day10

49 children across 21 parent sections.
Avg child size:  276 chars
Avg parent size: 572 chars

Example (one child + its parent):
  CHILD  (76 chars):  Acme Retail — Refund and Returns Policy
Effective 2024 | Customer Operations...
  PARENT (76 chars): Acme Retail — Refund and Returns Policy
Effective 2024 | Customer Operations...


## 3. Embed the children (not the parents)

The bi-encoder embeds small chunks. That's what gives small-to-big its retrieval *precision* — small chunks have focused semantics that match queries cleanly.

In [7]:
chunk_vectors = embed_texts(openai, [r["content"] for r in records], config["embedding_deployment"])
EMBED_DIM = len(chunk_vectors[0])
print(f"Embedded {len(chunk_vectors)} child chunks. Dim: {EMBED_DIM}")

Embedded 49 child chunks. Dim: 3072


## 4. Index with `parent_id` and `parent_content` fields

In [8]:
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField, SearchFieldDataType,
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
)

fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="content", type=SearchFieldDataType.String),         # child, embedded
    SearchableField(name="parent_content", type=SearchFieldDataType.String),  # parent, just stored
    SimpleField(name="parent_id", type=SearchFieldDataType.String, filterable=True),
    SimpleField(name="source",    type=SearchFieldDataType.String, filterable=True),
    SearchableField(name="section", type=SearchFieldDataType.String, filterable=True),
    SearchField(
        name="contentVector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=EMBED_DIM,
        vector_search_profile_name="vprofile",
    ),
]
vs = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw")],
    profiles=[VectorSearchProfile(name="vprofile", algorithm_configuration_name="hnsw")],
)

try:
    index_client.delete_index(INDEX_NAME)
    print(f"Deleted existing index '{INDEX_NAME}'.")
except Exception as e:
    print("No existing index (ok):", type(e).__name__)

index_client.create_index(SearchIndex(name=INDEX_NAME, fields=fields, vector_search=vs))
print(f"Index '{INDEX_NAME}' created with parent_id + parent_content fields.")

Deleted existing index 'rag-policies-day10'.
Index 'rag-policies-day10' created with parent_id + parent_content fields.


## 5. Upload

In [9]:
search = get_search_client(config, INDEX_NAME)
docs = [{**r, "contentVector": v} for r, v in zip(records, chunk_vectors)]
for i in range(0, len(docs), 100):
    search.upload_documents(documents=docs[i:i+100])
print(f"Uploaded {len(docs)} children.")

Uploaded 49 children.


## 6. The headline comparison — small-chunk RAG vs parent-doc RAG

Same question, same retrieval engine, but generation receives different-sized context. Small-chunk retrieval often gives a confidently incomplete answer when the exception lives in the next paragraph.

In [11]:
SELECT_CHILD  = ["content", "source", "section"]
SELECT_PARENT = ["content", "parent_content", "parent_id", "source", "section"]

def small_chunk_answer(question, k=4):
    hits = retrieve(search, openai, question, config["embedding_deployment"],
                    k=k, select=SELECT_CHILD)
    return answer(openai, config["chat_deployment"], question, hits, cite=True), hits

def parent_doc_answer(question, k_parents=3, k_children=15):
    hits = retrieve_parents(search, openai, question, config["embedding_deployment"],
                            k_children=k_children, k_parents=k_parents,
                            select=SELECT_PARENT)
    return answer(openai, config["chat_deployment"], question, hits, cite=True), hits

def compare(question, k=4):
    print("=" * 78)
    print(f"Q: {question}\n")

    a_small, hits_small = small_chunk_answer(question, k=k)
    print("-- SMALL-CHUNK RAG --")
    for i, h in enumerate(hits_small, 1):
        print(f"  [{i}] {h['source']} / {h['section']}")
        print(f"       '{h['content'][:90]}{'...' if len(h['content'])>90 else ''}'")
    print(f"\n  Answer:\n  {a_small}\n")

    a_parent, hits_parent = parent_doc_answer(question, k_parents=k)
    print("-- PARENT-DOC RAG --")
    for i, h in enumerate(hits_parent, 1):
        print(f"  [{i}] {h['source']} / {h['section']}  ({len(h['content'])} chars)")
        print(f"       triggered by child: '{h['matched_child'][:90]}{'...' if len(h['matched_child'])>90 else ''}'")
    print(f"\n  Answer:\n  {a_parent}\n")

## 7. Demo A — the "custom + damaged" exception

The matched sentence (*"custom items are generally not eligible for return"*) lives in section 3 of the refund policy. The exception (*"however, if a custom item arrives damaged, follow the defective items process"*) is the very next sentence in the same section. Small-chunk RAG often matches the rule and misses the exception; parent-doc RAG sees both because they share a parent.

In [12]:
compare("Can I return a custom-made item that arrived damaged?")

Q: Can I return a custom-made item that arrived damaged?

-- SMALL-CHUNK RAG --
  [1] 01_Refund_and_Returns_Policy.pdf / Custom or personalized orders
       'Custom-made and personalized items are generally not eligible for return or refund. This i...'
  [2] 01_Refund_and_Returns_Policy.pdf / Custom or personalized orders
       'e defective
items process described in section 4 below. The standard 30-day window does no...'
  [3] 01_Refund_and_Returns_Policy.pdf / Defective and damaged items
       'Items that arrive damaged in transit, or that fail within the manufacturer warranty period...'
  [4] 01_Refund_and_Returns_Policy.pdf / Bulk and corporate orders
       'not returnable except for
damage or defect, and damage must be reported within 7 days of d...'

  Answer:
  Yes, you can return a custom-made item that arrived damaged. According to the policy, damaged custom items are eligible for return or refund, but the damage must be reported within 7 days of delivery for the claim to 

## 8. Demo B — deletion timelines vs legal-hold exceptions

For deletion-of-data questions, the standard timeline lives at the top of section 5, but the legal-hold exception is the last paragraph of the same section. Parent-doc gives the complete answer.

In [13]:
compare("How long does it take to process my data deletion request?")

Q: How long does it take to process my data deletion request?

-- SMALL-CHUNK RAG --
  [1] 03_Data_Retention_and_Deletion_Policy.pdf / Data deletion requests
       'Customers may request deletion of their personal data through the privacy portal or by ema...'
  [2] 03_Data_Retention_and_Deletion_Policy.pdf / Data deletion requests
       'contractual obligations (such as transaction records under section 2) is retained for the ...'
  [3] 03_Data_Retention_and_Deletion_Policy.pdf / Backup retention
       'Daily backups are retained for 30 days. Weekly backups are retained for 13 weeks. Monthly
...'
  [4] 03_Data_Retention_and_Deletion_Policy.pdf / Introduction
       'Acme — Data Retention and Deletion Policy
Effective 2024 | Privacy and Legal'

  Answer:
  Based on the documents, data deletion requests are acknowledged within 5 business days and completed within 30 days, in line with GDPR Article 17 timelines [1].

-- PARENT-DOC RAG --
  [1] 03_Data_Retention_and_Deletion_Policy.pdf 

## 9. Demo C — nuanced disqualifiers

The disqualification logic spans two paragraphs in section 3 of the vendor onboarding doc — *"adverse findings don't automatically disqualify"* and *"sanctions matches are non-negotiable."* Small-chunk RAG can miss half the story.

In [14]:
compare("If a vendor has a negative finding in their background check, are they automatically rejected?")

Q: If a vendor has a negative finding in their background check, are they automatically rejected?

-- SMALL-CHUNK RAG --
  [1] 02_Vendor_Onboarding_Process.pdf / Background and compliance checks
       'before
re-screening.
Adverse findings do not automatically disqualify a vendor. The compli...'
  [2] 02_Vendor_Onboarding_Process.pdf / Background and compliance checks
       'All vendors pass through a standard background check that includes corporate registration
...'
  [3] 02_Vendor_Onboarding_Process.pdf / Background and compliance checks
       'data also undergo an information security questionnaire and a remote
evidence review of th...'
  [4] 02_Vendor_Onboarding_Process.pdf / Ongoing compliance reviews
       'All vendors are reviewed annually for continued compliance. Vendors handling customer data...'

  Answer:
  No, a vendor with a negative finding in their background check is not automatically rejected. The compliance team reviews each finding and may approve the vendor wi

## Day 10 checklist

1. Each indexed record carries `parent_id` and `parent_content`
2. `retrieve_parents()` returns deduped parent sections
3. On Demo A, parent-doc answer includes the damage exception that small-chunk answer omits
4. On Demo B, parent-doc answer mentions legal-hold exceptions
5. Parent context is large but well under context-window limits (3–5 parents of ~1500 chars each)

### Learnings
- **Precision and context are separate problems.** Retrieval needs precise (small) units; generation needs surrounding (large) context. Small-to-big decouples them.
- **The matched_child field is a great debugging signal.** When the parent-doc answer is right but the matched_child is from a different paragraph than you expected, that's a useful insight into how retrieval is actually behaving.
- **Cost stays roughly flat.** Same number of embeddings per query, same one LLM call — but the context window now carries 3–5× more useful text.